# Notebook 02 - Final Temporal Split and Dataset Audit v0.2

Purpose:
- Temporal split using dEven
- EMA_200 warm-up audit
- Train sufficiency audit
- Class balance audit
- Unseen test quality audit
- Censored observation audit
- Final universe selection

No model training is performed in this notebook.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
ROOT = Path(r"E:\Iran_stock_trade")

PREPARED_PATH = ROOT / "data_ready" / "prepared"
TRAIN_PATH = ROOT / "data_ready" / "train"
TEST_PATH = ROOT / "data_ready" / "unseen_test"
EVAL_PATH = ROOT / "data_ready" / "evaluation"

RESULT_PATH = ROOT / "results" / "tables"

for p in [TRAIN_PATH, TEST_PATH, RESULT_PATH]:
    p.mkdir(parents=True, exist_ok=True)


In [3]:
FEATURE_COLUMNS = [
"adj_open","adj_high","adj_low","adj_last_price",
"zigzag_up_new_2","zigzag_down_new_2",
"EMA_3","EMA_5","EMA_8","EMA_10","EMA_12","EMA_15",
"EMA_30","EMA_35","EMA_40","EMA_45","EMA_50",
"EMA_60","EMA_200","RSI_14","macd",
"power_of_buy","volume_per_avg_month",
"ho_buy","ho_sell","x","y","color",
"gmma","started","body"
]

WARMUP_PERIOD = 200

TRAIN_END = pd.Timestamp("2021-03-20")
TEST_START = pd.Timestamp("2021-03-21")
TEST_END = pd.Timestamp("2024-09-22")

In [4]:
files = list(PREPARED_PATH.glob("*_model.csv"))
print("Prepared files:", len(files))

Prepared files: 825


In [5]:
def convert_date(df):
    df["dEven"] = pd.to_datetime(
        df["dEven"].astype(str),
        format="%Y%m%d"
    )
    return df

In [6]:
# Temporal split
split_summary=[]

for file in files:
    df=pd.read_csv(file)
    df=convert_date(df)

    train=df[df["dEven"]<=TRAIN_END].copy()
    test=df[(df["dEven"]>=TEST_START)&(df["dEven"]<=TEST_END)].copy()

    symbol=file.stem.replace("_model","")

    train.to_csv(TRAIN_PATH/f"{symbol}_train.csv",index=False,encoding="utf-8-sig")
    test.to_csv(TEST_PATH/f"{symbol}_test.csv",index=False,encoding="utf-8-sig")

    split_summary.append({
        "symbol":symbol,
        "train_rows":len(train),
        "test_rows":len(test)
    })

split_summary=pd.DataFrame(split_summary)
split_summary.to_csv(RESULT_PATH/"temporal_split_summary.csv",index=False,encoding="utf-8-sig")

split_summary.head()

,symbol,train_rows,test_rows
0,آ س پ,1538,790
1,آباد,1594,742
2,آبادا,153,785
3,آبين,498,700
4,آردينه,0,111


In [7]:
# Warm-up audit and final universe creation

warmup=[]

for file in TRAIN_PATH.glob("*_train.csv"):
    df=pd.read_csv(file).sort_values("dEven")
    usable=df.iloc[WARMUP_PERIOD:]

    warmup.append({
        "symbol":file.stem.replace("_train",""),
        "usable_train_rows":len(usable),
        "class_1":(usable["class"]==1).sum(),
        "class_0":(usable["class"]==0).sum()
    })

warmup_df=pd.DataFrame(warmup)

warmup_df["class_1_ratio"]=warmup_df["class_1"]/warmup_df["usable_train_rows"]

warmup_df.to_csv(RESULT_PATH/"train_warmup_audit.csv",index=False,encoding="utf-8-sig")

warmup_df.head()

,symbol,usable_train_rows,class_1,class_0,class_1_ratio
0,آ س پ,1338,481,857,0.359492
1,آباد,1394,715,679,0.512912
2,آبادا,0,0,0,NaN
3,آبين,298,169,129,0.567114
4,آردينه,0,0,0,NaN


In [8]:
# Unseen test audit

test_reports=[]

for file in TEST_PATH.glob("*_test.csv"):
    df=pd.read_csv(file)

    missing=df[FEATURE_COLUMNS].isna().sum().sum()
    total=len(df)*len(FEATURE_COLUMNS)

    test_reports.append({
        "symbol":file.stem.replace("_test",""),
        "test_rows":len(df),
        "missing_rate":missing/total*100 if total else 0
    })

test_df=pd.DataFrame(test_reports)

test_df.to_csv(RESULT_PATH/"unseen_test_quality_audit.csv",index=False,encoding="utf-8-sig")

test_df.head()

,symbol,test_rows,missing_rate
0,آ س پ,790,0.000000
1,آباد,742,0.000000
2,آبادا,785,0.189028
3,آبين,700,0.000000
4,آردينه,111,15.257193


In [9]:
# Final universe

final_universe = warmup_df.merge(test_df,on="symbol")

final_universe = final_universe[
    (final_universe["usable_train_rows"]>=600) &
    (final_universe["test_rows"]>=500) &
    (final_universe["missing_rate"]<=5)
]

final_universe.to_csv(
    RESULT_PATH/"final_universe.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final universe size:", len(final_universe))

final_universe.head()

Final universe size: 470


,symbol,usable_train_rows,class_1,class_0,class_1_ratio,test_rows,missing_rate
0,آ س پ,1338,481,857,0.359492,790,0.0
1,آباد,1394,715,679,0.512912,742,0.0
8,آسيا,1370,618,752,0.451095,775,0.0
14,آپ,822,368,454,0.447689,785,0.0
16,اتكام,910,376,534,0.413187,757,0.0
